# TPC-H Collector Demo

# Missing
* connection data beyond loading
* join to connection data
* join to monitoring_single
* connection and configuration also in monitoring and connection df

## Naming
* There is an experiment having a code, say `1775855486`.
* The experiment inspects a SUT, say `PostgreSQL-A`. This is called a `configuration`.
* The experiment is run several times, say twice. The indicator of the run is called `experiment_run`.
* Each run can have several phases as a sequence. The number of the phase is called `client`. The state of the configuration in a phase is called a `connection`.
* Each client can have several `pods`, that are run in parallel. A pod represents a driver.
* Performance metrics are collected per driver pod.  
    The naming of an instance is `<sut>-<experiment_run>-<client>-<pod>`. It is unique per experiment.
* Monitoring metrics are collected per phase. They are automatically aggregated across parallel pods.  
    The naming of an instance is `<sut>-<experiment_run>-<client>`. It is unique per experiment.

## Aggregation
* Aggregation is complicated. Some metrics are aggregated via count, sum, max or average. Others cannot be aggregated sensibly, like experiment code or latency percentiles.
* There are helper functions to aggregated pods that are certainly run in parallel.  
  So `<sut>-<experiment_run>-<client>-<pod>` are aggregated to `<sut>-<experiment_run>-<client>`.
* An exception is multi-tenancy.

## Class `collector`
* constructor `collectors.benchbase(path, codes)`
* evaluator object `evaluate = collect.get_evaluator()`
* dataframe of connection infos `collect.get_connections()`
  * index is name of client
* **monitoring metrics**
  * dataframe of available metrics `collect.df_metrics`
    * index is key of metric
  * dataframe of monitored components `collect.get_monitored_phases()`
    * index is key of component
  * dataframe of aggregated metrics per connection `collect.get_monitoring_single_all()`
    * index is name of client
    * metrics aggregated per code, experiment_run and client
  * dataframe of aggregated metrics per connection and across tenants `collect.get_monitoring_all()`
    * index just enumerates
    * metrics aggregated per code, experiment_run and client and across tenants
  * dataframe of time series for a metric of a connection in an experiment `collect.get_monitoring_timeseries_single(code, metric)`
    * index is name of connection
    * unstacked (wide format)
  * dataframe of time series for a metric in all experiments `collect.get_monitoring_timeseries_all(metric)`
    * index just enumerates
    * stacked (long format)
* **performance metrics**
  * dataframe of loading metrics `collect.get_loading_time_max_all()`
    * index is name of client
  * dataframe of performance aggregated per parallel client `collect.get_performance_all()`
    * index just enumerates
    * performance aggregated per code, experiment_run and client
  * dataframe of performance for one experiment `collect.get_performance_single()`
    * performance of each single client (driver)
    * index is name of client pod
  * dataframe of performance for all experiments `collect.get_performance_all_single()`
    * performance of each single client (driver)
    * index is name of client pod


[1] [Benchmarking Multi-Tenant Architectures in PostgreSQL](https://doi.org/10.48786/edbt.2026.46)
> Erdelt, P.K., and Rabl T. (2026)
> In: Proceedings 29th International Conference on Extending Database Technology, EDBT 2026
> OpenProceedings.org
> https://doi.org/10.48786/edbt.2026.46


In [1]:
import pandas as pd
pd.set_option("display.max_rows", None)
pd.set_option('display.max_colwidth', None)

from bexhoma import collectors

%matplotlib inline

# Functions for Nice Plots

In [2]:
from notebookutils import *

# Collect Results

In [3]:
#path = r"D:\data\benchmarks"
path = r"/data/benchmarks"
filename_prefix = "demo_"
b_plot_save = False
b_skip_plots = True

In [4]:
codes = [
    "1776418956",
    "1776420958"
]

codes

['1776418956', '1776420958']

In [5]:
collect = collectors.default(path, codes)

# Get all Metrics Metadata

## Metrics Names and Types

Metrics that are derived from monitoring

In [6]:
collect.df_metrics

,title,active,type,metric
total_cpu_memory,Memory Usage [MiB],True,cluster,gauge
total_cpu_memory_cached,Memory Usage Cached [MiB],True,cluster,gauge
total_cpu_util,CPU Utilization,True,cluster,gauge
total_cpu_throttled,CPU Throttle,True,cluster,gauge
total_cpu_throttled_s,CPU Throttled Time [s],True,cluster,counter
total_cpu_util_others,CPU Utilization Others,False,cluster,gauge
total_cpu_util_s,CPU Utilization Time [s],True,cluster,counter
total_cpu_util_user_s,CPU User Time [s],True,cluster,counter
total_cpu_util_sys_s,CPU System Time [s],True,cluster,counter
total_cpu_util_others_s,CPU Utilization Time Others [s],False,cluster,counter


## Names of Monitored Phases

Names of components and their phases

In [7]:
collect.get_monitored_phases()

,description
loading,Loading phase: SUT deployment
datagenerator,Loading phase: component data generator
loader,Loading phase: component loader
stream,Execution phase: SUT deployment
benchmarker,Execution phase: component benchmarker


# Get Connection Infos

List of states of SUTs, containing loading infos.

In [8]:
df = collect.get_connections()
df.T

,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-2-2
code,1776418956,1776418956,1776418956,1776418956,1776420958,1776420958,1776420958,1776420958
experiment_run,1,1,2,2,1,1,2,2
client,1,2,1,2,1,2,1,2
dockerimage,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3,postgres:18.3
connection,PostgreSQL-BHT-8-1-1-1,PostgreSQL-BHT-8-1-2-2,PostgreSQL-BHT-8-2-1-1,PostgreSQL-BHT-8-2-2-2,PostgreSQL-BHT-8-1-1-1,PostgreSQL-BHT-8-1-2-2,PostgreSQL-BHT-8-2-1-1,PostgreSQL-BHT-8-2-2-2
time_load,455.0,455.0,455.0,455.0,1256.0,1256.0,1256.0,1256.0
time_ingest,141.0,141.0,141.0,141.0,350.0,350.0,350.0,350.0
time_check,291.0,291.0,291.0,291.0,843.0,843.0,843.0,843.0
terminals,0,0,0,0,0,0,0,0
pods,1,2,1,2,1,2,1,2


# Get Aggregated Metrics per SUT and per Experiment

In [9]:
df_performance = collect.get_monitoring_aggregated_per_connection("stream")
df_performance.T

,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-2-2
Memory Usage [MiB],10919.47,12131.24,10392.28,11955.10,17504.07,19192.60,14054.22,16993.59
Memory Usage Cached [MiB],15835.40,17047.17,15288.83,16789.84,27302.43,28990.96,22555.50,26276.44
CPU Utilization,1.66,3.29,0.96,3.39,2.91,5.70,1.60,5.64
CPU Throttle,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
CPU Throttled Time [s],0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
CPU Utilization Time [s],180.14,320.88,998.63,395.04,381.59,869.22,2263.06,877.07
CPU User Time [s],156.36,281.35,826.53,344.83,331.65,767.27,1928.57,768.57
CPU System Time [s],23.78,39.53,172.10,50.21,49.94,101.95,334.50,108.51
Network Rx Total [MiB],1.72,7.30,3307.36,1.87,1.91,3.74,6644.39,3.68
Network Tx Total [MiB],0.20,1320.29,21.81,3.35,1.96,3.78,39.78,3.66


In [10]:
df_performance = collect.add_metadata(df_performance)
df_performance.T

combine on index


,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2,1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-2-2
Memory Usage [MiB],10919.47,12131.24,10392.28,11955.1,17504.07,19192.6,14054.22,16993.59
Memory Usage Cached [MiB],15835.4,17047.17,15288.83,16789.84,27302.43,28990.96,22555.5,26276.44
CPU Utilization,1.66,3.29,0.96,3.39,2.91,5.7,1.6,5.64
CPU Throttle,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
CPU Throttled Time [s],0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
CPU Utilization Time [s],180.14,320.88,998.63,395.04,381.59,869.22,2263.06,877.07
CPU User Time [s],156.36,281.35,826.53,344.83,331.65,767.27,1928.57,768.57
CPU System Time [s],23.78,39.53,172.1,50.21,49.94,101.95,334.5,108.51
Network Rx Total [MiB],1.72,7.3,3307.36,1.87,1.91,3.74,6644.39,3.68
Network Tx Total [MiB],0.2,1320.29,21.81,3.35,1.96,3.78,39.78,3.66


# Bar Plots of Aggregated Values per Metric

In [11]:
metric = 'total_cpu_memory'
code = codes[0]
df = collect.get_monitoring_timeseries_per_connection(code, metric=metric)
df.T

,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2
0,9770.968750,10315.593750,10315.644531,10300.511719
1,9770.968750,10315.593750,10315.644531,10300.511719
2,9770.968750,10315.593750,10315.644531,10300.511719
3,9770.968750,10315.593750,10315.644531,10300.511719
4,9770.968750,10315.593750,10315.644531,10300.511719
5,9838.882812,10315.593750,10315.644531,10300.511719
6,9838.882812,10315.593750,10315.644531,10300.511719
7,9838.882812,10315.593750,10315.644531,10300.511719
8,9838.882812,10315.593750,10315.644531,10300.511719
9,9838.882812,10315.593750,10315.644531,10300.511719


In [12]:
df_performance = collect.add_metadata(df)
df_performance.T

combine on index


,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2
0,9770.96875,10315.59375,10315.644531,10300.511719
1,9770.96875,10315.59375,10315.644531,10300.511719
2,9770.96875,10315.59375,10315.644531,10300.511719
3,9770.96875,10315.59375,10315.644531,10300.511719
4,9770.96875,10315.59375,10315.644531,10300.511719
5,9838.882812,10315.59375,10315.644531,10300.511719
6,9838.882812,10315.59375,10315.644531,10300.511719
7,9838.882812,10315.59375,10315.644531,10300.511719
8,9838.882812,10315.59375,10315.644531,10300.511719
9,9838.882812,10315.59375,10315.644531,10300.511719


In [13]:
metric = 'total_cpu_memory'
code = codes[0]
collect.get_monitoring_timeseries_all(metric=metric)

,timestamp,code,experiment_run,client,type,vol_tenants,num_tenants,value
0,0,1776418956,1,1,None,False,0,9770.968750
1,0,1776418956,1,2,None,False,0,10315.593750
2,0,1776418956,2,1,None,False,0,10315.644531
3,0,1776418956,2,2,None,False,0,10300.511719
4,1,1776418956,1,1,None,False,0,9770.968750
5,1,1776418956,1,2,None,False,0,10315.593750
6,1,1776418956,2,1,None,False,0,10315.644531
7,1,1776418956,2,2,None,False,0,10300.511719
8,2,1776418956,1,1,None,False,0,9770.968750
9,2,1776418956,1,2,None,False,0,10315.593750


In [16]:
df_performance = collect.get_performance_aggregated_per_connection()

df_performance_first = df_performance[df_performance['client']==1]
df_performance_second = df_performance[df_performance['client']==2]

df_performance.dropna(inplace=True)
df_performance

,connection,experiment_run,total_timer_execution,Power@Size [~Q/h],code,orig_name,SF,time [s],count,client,Throughput@Size,num_of_queries
1776418956-PostgreSQL-BHT-8-1-1,PostgreSQL-BHT-8-1-1,1,1.566479,6894.444445,1776418956,PostgreSQL-BHT-8-1-1,3.0,58,1,1,4096.55,22
1776418956-PostgreSQL-BHT-8-1-2,PostgreSQL-BHT-8-1-2,1,1.573540,6863.507120,1776418956,PostgreSQL-BHT-8-1-2,3.0,60,2,2,7920.00,44
1776418956-PostgreSQL-BHT-8-2-1,PostgreSQL-BHT-8-2-1,2,2.345645,4604.277867,1776418956,PostgreSQL-BHT-8-2-1,3.0,120,1,1,1980.00,22
1776418956-PostgreSQL-BHT-8-2-2,PostgreSQL-BHT-8-2-2,2,1.588448,6799.089476,1776418956,PostgreSQL-BHT-8-2-2,3.0,60,2,2,7920.00,44
1776420958-PostgreSQL-BHT-8-1-1,PostgreSQL-BHT-8-1-1,1,2.913431,7413.940216,1776420958,PostgreSQL-BHT-8-1-1,6.0,113,1,1,4205.31,22
1776420958-PostgreSQL-BHT-8-1-2,PostgreSQL-BHT-8-1-2,1,2.921044,7394.616680,1776420958,PostgreSQL-BHT-8-1-2,6.0,110,2,2,8640.00,44
1776420958-PostgreSQL-BHT-8-2-1,PostgreSQL-BHT-8-2-1,2,4.244649,5088.759517,1776420958,PostgreSQL-BHT-8-2-1,6.0,210,1,1,2262.86,22
1776420958-PostgreSQL-BHT-8-2-2,PostgreSQL-BHT-8-2-2,2,2.952376,7316.141092,1776420958,PostgreSQL-BHT-8-2-2,6.0,119,2,2,7986.55,44


In [15]:
df_performance = collect.get_performance_per_pod()

df_performance_first = df_performance[df_performance['client']==1]
df_performance_second = df_performance[df_performance['client']==2]

df_performance.T

,1776418956-PostgreSQL-BHT-8-1-1-1,1776418956-PostgreSQL-BHT-8-1-2-1,1776418956-PostgreSQL-BHT-8-1-2-2,1776418956-PostgreSQL-BHT-8-2-1-1,1776418956-PostgreSQL-BHT-8-2-2-1,1776418956-PostgreSQL-BHT-8-2-2-2,1776420958-PostgreSQL-BHT-8-1-1-1,1776420958-PostgreSQL-BHT-8-1-2-1,1776420958-PostgreSQL-BHT-8-1-2-2,1776420958-PostgreSQL-BHT-8-2-1-1,1776420958-PostgreSQL-BHT-8-2-2-1,1776420958-PostgreSQL-BHT-8-2-2-2
total_timer_execution,1.566479,1.597086,1.55034,2.345645,1.588409,1.588487,2.913431,2.934896,2.907257,4.244649,2.986872,2.918278
Power@Size [~Q/h],6894.444445,6762.31504,6966.213449,4604.277867,6799.256171,6798.922785,7413.940216,7359.714573,7429.684303,5088.759517,7231.64457,7401.624895
Geo Times [s],1.566479,1.597086,1.55034,2.345645,1.588409,1.588487,2.913431,2.934896,2.907257,4.244649,2.986872,2.918278
orig_name,PostgreSQL-BHT-8-1-1,PostgreSQL-BHT-8-1-2,PostgreSQL-BHT-8-1-2,PostgreSQL-BHT-8-2-1,PostgreSQL-BHT-8-2-2,PostgreSQL-BHT-8-2-2,PostgreSQL-BHT-8-1-1,PostgreSQL-BHT-8-1-2,PostgreSQL-BHT-8-1-2,PostgreSQL-BHT-8-2-1,PostgreSQL-BHT-8-2-2,PostgreSQL-BHT-8-2-2
SF,3.0,3.0,3.0,3.0,3.0,3.0,6.0,6.0,6.0,6.0,6.0,6.0
experiment_run,1,1,1,2,2,2,1,1,1,2,2,2
client,1,2,2,1,2,2,1,2,2,1,2,2
time [s],58,60,57,120,59,60,113,110,110,210,119,116
count,1,1,1,1,1,1,1,1,1,1,1,1
connection,PostgreSQL-BHT-8-1-1,PostgreSQL-BHT-8-1-2,PostgreSQL-BHT-8-1-2,PostgreSQL-BHT-8-2-1,PostgreSQL-BHT-8-2-2,PostgreSQL-BHT-8-2-2,PostgreSQL-BHT-8-1-1,PostgreSQL-BHT-8-1-2,PostgreSQL-BHT-8-1-2,PostgreSQL-BHT-8-2-1,PostgreSQL-BHT-8-2-2,PostgreSQL-BHT-8-2-2


In [17]:
df = collect.add_metadata(df_performance)
df.T


combine on columns
df Duplikate: False
df_connections Duplikate: False


connection_pod,1776418956-PostgreSQL-BHT-8-1-1,1776418956-PostgreSQL-BHT-8-1-2,1776418956-PostgreSQL-BHT-8-2-1,1776418956-PostgreSQL-BHT-8-2-2,NaN,NaN,NaN,NaN,1776420958-PostgreSQL-BHT-8-1-1,1776420958-PostgreSQL-BHT-8-1-2,1776420958-PostgreSQL-BHT-8-2-1,1776420958-PostgreSQL-BHT-8-2-2,NaN,NaN,NaN,NaN
code,1776418956,1776418956,1776418956,1776418956,1776418956,1776418956,1776418956,1776418956,1776420958,1776420958,1776420958,1776420958,1776420958,1776420958,1776420958,1776420958
connection,PostgreSQL-BHT-8-1-1,PostgreSQL-BHT-8-1-2,PostgreSQL-BHT-8-2-1,PostgreSQL-BHT-8-2-2,PostgreSQL-BHT-8-1-1-1,PostgreSQL-BHT-8-1-2-2,PostgreSQL-BHT-8-2-1-1,PostgreSQL-BHT-8-2-2-2,PostgreSQL-BHT-8-1-1,PostgreSQL-BHT-8-1-2,PostgreSQL-BHT-8-2-1,PostgreSQL-BHT-8-2-2,PostgreSQL-BHT-8-1-1-1,PostgreSQL-BHT-8-1-2-2,PostgreSQL-BHT-8-2-1-1,PostgreSQL-BHT-8-2-2-2
experiment_run,1.0,1.0,2.0,2.0,NaN,NaN,NaN,NaN,1.0,1.0,2.0,2.0,NaN,NaN,NaN,NaN
total_timer_execution,1.566479,1.57354,2.345645,1.588448,NaN,NaN,NaN,NaN,2.913431,2.921044,4.244649,2.952376,NaN,NaN,NaN,NaN
Power@Size [~Q/h],6894.444445,6863.50712,4604.277867,6799.089476,NaN,NaN,NaN,NaN,7413.940216,7394.61668,5088.759517,7316.141092,NaN,NaN,NaN,NaN
orig_name,PostgreSQL-BHT-8-1-1,PostgreSQL-BHT-8-1-2,PostgreSQL-BHT-8-2-1,PostgreSQL-BHT-8-2-2,NaN,NaN,NaN,NaN,PostgreSQL-BHT-8-1-1,PostgreSQL-BHT-8-1-2,PostgreSQL-BHT-8-2-1,PostgreSQL-BHT-8-2-2,NaN,NaN,NaN,NaN
SF,3.0,3.0,3.0,3.0,NaN,NaN,NaN,NaN,6.0,6.0,6.0,6.0,NaN,NaN,NaN,NaN
time [s],58.0,60.0,120.0,60.0,NaN,NaN,NaN,NaN,113.0,110.0,210.0,119.0,NaN,NaN,NaN,NaN
count,1.0,2.0,1.0,2.0,NaN,NaN,NaN,NaN,1.0,2.0,1.0,2.0,NaN,NaN,NaN,NaN
client,1.0,2.0,1.0,2.0,NaN,NaN,NaN,NaN,1.0,2.0,1.0,2.0,NaN,NaN,NaN,NaN


In [22]:
from scipy.stats import gmean

#df = collect.get_evaluator().evaluation.get_aggregated_query_statistics(type='latency', name='execution', query_aggregate='Mean')
#if not df is None:
#    df = df.sort_index().T.round(2)
#    df.index = df.index.map(map_index_to_queryname)
#    num_of_queries = len(df.index)

num_of_queries=22
df=df_performance.copy()
column = "connection"
df_aggregated = pd.DataFrame()
for key, grp in df.groupby(column):
    #print(key, len(grp.index))
    #print(grp.columns)
    aggregate = {
        'total_timer_execution':gmean,
        'Power@Size [~Q/h]':gmean,
        'code':'max',
        'orig_name':'max',
        'SF':'max',
        'experiment_run':'max',
        'time [s]':'max',
        'count':'count',
        'Throughput@Size':'max',
        'num_of_queries':'mean',
    }
    dict_grp = dict()
    dict_grp['connection'] = key
    #dict_grp['configuration'] = grp['configuration'].iloc[0]
    dict_grp['experiment_run'] = grp['experiment_run'].iloc[0]
    #dict_grp['client'] = grp['client'][0]
    #dict_grp['pod'] = grp['pod'][0]
    #print(dict_grp)
    dict_grp = {**dict_grp, **grp.agg(aggregate)}
    df_grp = pd.DataFrame(dict_grp, index=[key])#columns=list(dict_grp.keys()))
    #df_grp = df_grp.T
    #df_grp.set_index('connection', inplace=True)
    #print(df_grp)
    df_aggregated = pd.concat([df_aggregated, df_grp])

df_aggregated['Throughput@Size'] = (df_aggregated['num_of_queries']*3600.*df_aggregated['count']/df_aggregated['time [s]']*df_aggregated['SF']).round(2)
df_aggregated

,connection,experiment_run,total_timer_execution,Power@Size [~Q/h],code,orig_name,SF,time [s],count,Throughput@Size,num_of_queries
PostgreSQL-BHT-8-1-1,PostgreSQL-BHT-8-1-1,1,2.136311,7149.475431,1776420958,PostgreSQL-BHT-8-1-1,6.0,113,2,8410.62,22.0
PostgreSQL-BHT-8-1-2,PostgreSQL-BHT-8-1-2,1,2.143917,7124.114277,1776420958,PostgreSQL-BHT-8-1-2,6.0,110,4,17280.00,22.0
PostgreSQL-BHT-8-2-1,PostgreSQL-BHT-8-2-1,2,3.155383,4840.461013,1776420958,PostgreSQL-BHT-8-2-1,6.0,210,2,4525.71,22.0
PostgreSQL-BHT-8-2-2,PostgreSQL-BHT-8-2-2,2,2.165571,7052.878696,1776420958,PostgreSQL-BHT-8-2-2,6.0,119,4,15973.11,22.0


In [16]:
cols = ['code', 'connection']
print(df_performance.index.duplicated())
df = df_performance.rename_axis('connection_pod').reset_index()
df = df.set_index(cols)
df

[False False False False False False False False False False False False]


connection_pod  \
code       connection                                                
1776418956 PostgreSQL-BHT-8-1-1  1776418956-PostgreSQL-BHT-8-1-1-1   
           PostgreSQL-BHT-8-1-2  1776418956-PostgreSQL-BHT-8-1-2-1   
           PostgreSQL-BHT-8-1-2  1776418956-PostgreSQL-BHT-8-1-2-2   
           PostgreSQL-BHT-8-2-1  1776418956-PostgreSQL-BHT-8-2-1-1   
           PostgreSQL-BHT-8-2-2  1776418956-PostgreSQL-BHT-8-2-2-1   
           PostgreSQL-BHT-8-2-2  1776418956-PostgreSQL-BHT-8-2-2-2   
1776420958 PostgreSQL-BHT-8-1-1  1776420958-PostgreSQL-BHT-8-1-1-1   
           PostgreSQL-BHT-8-1-2  1776420958-PostgreSQL-BHT-8-1-2-1   
           PostgreSQL-BHT-8-1-2  1776420958-PostgreSQL-BHT-8-1-2-2   
           PostgreSQL-BHT-8-2-1  1776420958-PostgreSQL-BHT-8-2-1-1   
           PostgreSQL-BHT-8-2-2  1776420958-PostgreSQL-BHT-8-2-2-1   
           PostgreSQL-BHT-8-2-2  1776420958-PostgreSQL-BHT-8-2-2-2   

                                 total_timer_execution  Power@Size [~Q/h]  \
code       connection                                                       
1776418956 PostgreSQL-BHT-8-1-1               1.566479        6894.444445   
           PostgreSQL-BHT-8-1-2               1.597086        6762.315040   
           PostgreSQL-BHT-8-1-2               1.550340        6966.213449   
           PostgreSQL-BHT-8-2-1               2.345645        4604.277867   
           PostgreSQL-BHT-8-2-2               1.588409        6799.256171   
           PostgreSQL-BHT-8-2-2               1.588487        6798.922785   
1776420958 PostgreSQL-BHT-8-1-1               2.913431        7413.940216   
           PostgreSQL-BHT-8-1-2               2.934896        7359.714573   
           PostgreSQL-BHT-8-1-2               2.907257        7429.684303   
           PostgreSQL-BHT-8-2-1               4.244649        5088.759517   
           PostgreSQL-BHT-8-2-2               2.986872        7231.644570   
           PostgreSQL-BHT-8-2-2               2.918278        7401.624895   

                                 Geo Times [s]             orig_name   SF  \
code       connection                                                       
1776418956 PostgreSQL-BHT-8-1-1       1.566479  PostgreSQL-BHT-8-1-1  3.0   
           PostgreSQL-BHT-8-1-2       1.597086  PostgreSQL-BHT-8-1-2  3.0   
           PostgreSQL-BHT-8-1-2       1.550340  PostgreSQL-BHT-8-1-2  3.0   
           PostgreSQL-BHT-8-2-1       2.345645  PostgreSQL-BHT-8-2-1  3.0   
           PostgreSQL-BHT-8-2-2       1.588409  PostgreSQL-BHT-8-2-2  3.0   
           PostgreSQL-BHT-8-2-2       1.588487  PostgreSQL-BHT-8-2-2  3.0   
1776420958 PostgreSQL-BHT-8-1-1       2.913431  PostgreSQL-BHT-8-1-1  6.0   
           PostgreSQL-BHT-8-1-2       2.934896  PostgreSQL-BHT-8-1-2  6.0   
           PostgreSQL-BHT-8-1-2       2.907257  PostgreSQL-BHT-8-1-2  6.0   
           PostgreSQL-BHT-8-2-1       4.244649  PostgreSQL-BHT-8-2-1  6.0   
           PostgreSQL-BHT-8-2-2       2.986872  PostgreSQL-BHT-8-2-2  6.0   
           PostgreSQL-BHT-8-2-2       2.918278  PostgreSQL-BHT-8-2-2  6.0   

                                 experiment_run  client  time [s]  count  \
code       connection                                                      
1776418956 PostgreSQL-BHT-8-1-1               1       1        58      1   
           PostgreSQL-BHT-8-1-2               1       2        60      1   
           PostgreSQL-BHT-8-1-2               1       2        57      1   
           PostgreSQL-BHT-8-2-1               2       1       120      1   
           PostgreSQL-BHT-8-2-2               2       2        59      1   
           PostgreSQL-BHT-8-2-2               2       2        60      1   
1776420958 PostgreSQL-BHT-8-1-1               1       1       113      1   
           PostgreSQL-BHT-8-1-2               1       2       110      1   
           PostgreSQL-BHT-8-1-2               1       2       110      1   
           PostgreSQL-BHT-8-2-1               2       1       210      1   
 

In [17]:
df_connections = collect.get_connections()
df_connections = df_connections.set_index(cols)
df_connections

experiment_run client    dockerimage  \
code       connection                                                    
1776418956 PostgreSQL-BHT-8-1-1-1              1      1  postgres:18.3   
           PostgreSQL-BHT-8-1-2-2              1      2  postgres:18.3   
           PostgreSQL-BHT-8-2-1-1              2      1  postgres:18.3   
           PostgreSQL-BHT-8-2-2-2              2      2  postgres:18.3   
1776420958 PostgreSQL-BHT-8-1-1-1              1      1  postgres:18.3   
           PostgreSQL-BHT-8-1-2-2              1      2  postgres:18.3   
           PostgreSQL-BHT-8-2-1-1              2      1  postgres:18.3   
           PostgreSQL-BHT-8-2-2-2              2      2  postgres:18.3   

                                  time_load time_ingest time_check terminals  \
code       connection                                                          
1776418956 PostgreSQL-BHT-8-1-1-1     455.0       141.0      291.0         0   
           PostgreSQL-BHT-8-1-2-2     455.0       141.0      291.0         0   
           PostgreSQL-BHT-8-2-1-1     455.0       141.0      291.0         0   
           PostgreSQL-BHT-8-2-2-2     455.0       141.0      291.0         0   
1776420958 PostgreSQL-BHT-8-1-1-1    1256.0       350.0      843.0         0   
           PostgreSQL-BHT-8-1-2-2    1256.0       350.0      843.0         0   
           PostgreSQL-BHT-8-2-1-1    1256.0       350.0      843.0         0   
           PostgreSQL-BHT-8-2-2-2    1256.0       350.0      843.0         0   

                                  pods tenant num_worker  ... arg_autovacuum  \
code       connection                                     ...                  
1776418956 PostgreSQL-BHT-8-1-1-1    1                 0  ...            off   
           PostgreSQL-BHT-8-1-2-2    2                 0  ...            off   
           PostgreSQL-BHT-8-2-1-1    1                 0  ...            off   
           PostgreSQL-BHT-8-2-2-2    2                 0  ...            off   
1776420958 PostgreSQL-BHT-8-1-1-1    1                 0  ...            off   
           PostgreSQL-BHT-8-1-2-2    2                 0  ...            off   
           PostgreSQL-BHT-8-2-1-1    1                 0  ...            off   
           PostgreSQL-BHT-8-2-2-2    2                 0  ...            off   

                                  arg_wal_level arg_max_wal_senders arg_fsync  \
code       connection                                                           
1776418956 PostgreSQL-BHT-8-1-1-1       minimal                   0        on   
           PostgreSQL-BHT-8-1-2-2       minimal                   0        on   
           PostgreSQL-BHT-8-2-1-1       minimal                   0        on   
           PostgreSQL-BHT-8-2-2-2       minimal                   0        on   
1776420958 PostgreSQL-BHT-8-1-1-1       minimal                   0        on   
           PostgreSQL-BHT-8-1-2-2       minimal                   0        on   
           PostgreSQL-BHT-8-2-1-1       minimal                   0        on   
           PostgreSQL-BHT-8-2-2-2       minimal                   0        on   

                                  arg_wal_compression arg_synchronous_commit  \
code       connection                                                          
1776418956 PostgreSQL-BHT-8-1-1-1                  on                     on   
           PostgreSQL-BHT-8-1-2-2                  on                     on   
           PostgreSQL-BHT-8-2-1-1                  on                     on   
           PostgreSQL-BHT-8-2-2-2                  on                     on   
1776420958 PostgreSQL-BHT-8-1-1-1                  on                     on   
           PostgreSQL-BHT-8-1-2-2                  on                     on   
           PostgreSQL-BHT-8-2-1-1                  on                     on   
           PostgreSQL-BHT-8-2-2-2                  on                     on   

                                  arg_max_wal_size arg_min_wal_size  \
code    

In [18]:
cols_to_use = [c for c in df_connections.columns if c not in df.columns]
cols_to_use

['dockerimage',
 'time_load',
 'time_ingest',
 'time_check',
 'terminals',
 'pods',
 'tenant',
 'num_worker',
 'host_RAM',
 'host_CPU',
 'host_GPU',
 'host_Cores',
 'host_host',
 'host_node',
 'host_disk',
 'host_datadisk',
 'host_volume_size',
 'host_volume_used',
 'host_cuda',
 'host_hugepages_total',
 'host_hugepages_free',
 'host_cpu_list',
 'host_requests_cpu',
 'host_requests_memory',
 'host_limits_cpu',
 'host_limits_memory',
 'loading_parameters_SF',
 'loading_parameters_PODS_TOTAL',
 'loading_parameters_PODS_PARALLEL',
 'loading_parameters_STORE_RAW_DATA',
 'loading_parameters_STORE_RAW_DATA_RECREATE',
 'loading_parameters_BEXHOMA_SYNCH_LOAD',
 'loading_parameters_BEXHOMA_SYNCH_GENERATE',
 'loading_parameters_TRANSFORM_RAW_DATA',
 'loading_parameters_TPCH_TABLE',
 'loading_parameters_BEXHOMA_TENANT_BY',
 'loading_parameters_BEXHOMA_TENANT_NUM',
 'benchmarking_parameters_SF',
 'benchmarking_parameters_DBMSBENCHMARKER_RECREATE_PARAMETER',
 'benchmarking_parameters_DBMSBENCHMARKE

In [19]:
result = pd.concat([df, df_connections[cols_to_use]], axis=1)

InvalidIndexError: Reindexing only valid with uniquely valued Index objects

In [ ]:
df = collect.add_metadata(df_performance)
df.T


In [20]:
df_performance = collect.get_performance_aggregated_per_connection()

df_performance_first = df_performance[df_performance['client']==1]
df_performance_second = df_performance[df_performance['client']==2]

df_performance.dropna(inplace=True)

NameError: name 'collect' is not defined

# Boxplot of All Metrics for a Single Experiment

In [ ]:
if not b_skip_plots:
    results = []
    code = codes[0]
    for idx, row in collect.df_metrics.iterrows():
        if row["active"] == False:
            continue
        metric_name = idx
        method = 'diff' if row["metric"] == 'counter' else 'mean'
        col_name = row["title"]
        df_monitoring = collect.get_monitoring_timeseries_single(code, metric=metric_name)
        #print(df_monitoring)
        ax = df_monitoring.boxplot()
        ax.set_title(col_name)
        plt.show()

# Boxplot of A Single Metric for a Single Experiment

In [ ]:
metric = 'total_cpu_memory'
code = codes[0]
collect.get_monitoring_timeseries_single(code, metric=metric)

In [ ]:
metric = 'total_cpu_memory'
code = codes[0]
collect.get_monitoring_timeseries_all( metric=metric)

In [ ]:
if not b_skip_plots:
    metric = 'pg_stat_activity_max_tx_duration_active'
    #metric = 'pg_stat_database_blks_reads'
    #metric = 'pg_stat_activity_count_idle_transaction'
    metric = 'total_cpu_memory'
    code = codes[0]
    df_monitoring = collect.get_monitoring_timeseries_single(code, metric=metric)
    ax = df_monitoring.boxplot()
    ax.set_title(collect.df_metrics.loc[metric]['title'])#metric)
    plt.show()
    df_monitoring

# Lineplot of a Single Metric for a Single Experiment

In [ ]:
if not b_skip_plots:
    code = codes[0]
    metric = 'pg_stat_database_blks_hit'
    metric = 'pg_stat_activity_count_idle_transaction'
    metric = 'total_cpu_memory'
    metric = 'total_cpu_util'
    metric = 'total_cpu_util_s'
    metric = 'core_variance'
    df_monitoring = collect.get_monitoring_timeseries_single(code, metric=metric)
    ax = df_monitoring.plot()
    ax.set_title(metric)
    plt.show()
    #df_monitoring

# Boxplot of a Single Metric for all Experiments

In [ ]:
metric = 'pg_stat_database_blks_hit'
metric = 'pg_stat_activity_count_idle_transaction'
metric = 'total_cpu_memory'
metric = 'total_cpu_util'
metric = 'total_cpu_util_s'
metric = 'core_variance'
metric = 'pg_locks_count_exclusivelock'
metric = 'pg_stat_activity_count_active'
df_performance = collect.get_monitoring_timeseries_all(metric)
#df_performance
df_performance_first = df_performance[df_performance['client']=="1"]
#df_performance_first
df_performance_second = df_performance[df_performance['client']=="2"]
#df_performance_second
#df_performance_first['value'].describe()
#df_performance

## First Execution Run

In [ ]:
plot_boxplots(df_performance_first, y='value', title=collect.df_metrics.loc[metric]['title'], b_plot_save=b_plot_save, filename_prefix=filename_prefix)

## Second Execution Run

In [ ]:
plot_boxplots(df_performance_second, y='value', title=collect.df_metrics.loc[metric]['title'], b_plot_save=b_plot_save, filename_prefix=filename_prefix)

# Lineplot of a Single Metric for All Experiments

## First Execution Run

In [ ]:
#df = df_performance_first[df_performance_first['type'] == 'None']
#df = df[df['experiment_run'] == '1']
plot_lines(df_performance_first, y='value', title=collect.df_metrics.loc[metric]['title'], b_plot_save=b_plot_save, filename_prefix=filename_prefix)
#df

# Aggregated Metrics for all Connections

In [ ]:
df_performance = collect.get_monitoring_single_all("stream")
df_performance.T

# Monitoring Aggregated Values

In [ ]:
df_performance = collect.get_monitoring_all("stream")

df_performance_first = df_performance[df_performance['client']==1]
df_performance_second = df_performance[df_performance['client']==2]

df_performance.T#[['Max CPU', 'client', 'type', 'num_tenants']]

## First Execution Run

In [ ]:
plot_bars(df_performance_first, y='Buffer Cache Hit Ratio', title='Cache Hit Ratio [%]', estimator='min', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

## Second Execution Run

In [ ]:
plot_bars(df_performance_second, y='Buffer Cache Hit Ratio', title='Cache Hit Ratio [%]', estimator='min', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

# Performance Results per Tenant

In [ ]:
df_performance = collect.get_performance_all_single()

df_performance_first = df_performance[df_performance['client']==1]
df_performance_second = df_performance[df_performance['client']==2]

df_performance.T

## First Execution Run

In [ ]:
plot_boxplots(df_performance_first, y='Goodput (requests/second)', title='Distribution of Goodput First Run [req/s]', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_boxplots(df_performance_second, y='Goodput (requests/second)', title='Distribution of Goodput Second Run [req/s]', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_boxplots(df_performance_first, y='Latency Distribution.99th Percentile Latency (microseconds)', title=r'Distribution of 99th Latency 1st Run [$\mu s$]', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_boxplots(df_performance_second, y='Latency Distribution.99th Percentile Latency (microseconds)', title=r'Distribution of 99th Latency 2nd Run [$\mu s$]', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

# Performance Results per Total

In [ ]:
collect.get_performance_single().T

In [ ]:
collect.get_performance_all_single().T

In [ ]:
df_performance = collect.get_performance_all()

df_performance_first = df_performance[df_performance['client']==1]
df_performance_second = df_performance[df_performance['client']==2]

df_performance.dropna(inplace=True)

In [ ]:
df_performance.T

In [ ]:
import seaborn as sns

sns.barplot(data=df_performance, x='experiment_run', y='Goodput (requests/second)', hue='client')
plt.show()


In [ ]:
plot_bars(df_performance, y='Goodput (requests/second)', title='Goodput [req/s]', estimator='min', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance_first, y='Goodput (requests/second)', title='Goodput First Run [req/s]', estimator='min', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance_second, y='Goodput (requests/second)', title='Goodput Second Run [req/s]', estimator='min', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance, y='Latency Distribution.Average Latency (microseconds)', title=r'Average Latency [$\mu s$]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance_first, y='Latency Distribution.Average Latency (microseconds)', title=r'Average Latency First Run [$\mu s$]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance_second, y='Latency Distribution.Average Latency (microseconds)', title=r'Average Latency Second Run [$\mu s$]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance, y='num_errors', title='Deadlocks', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
df_performance = collect.get_loading_time_max_all()

df_performance_first = df_performance[df_performance['client']==1]
df_performance_second = df_performance[df_performance['client']==2]

df_performance

In [ ]:
df_performance_first = df_performance[df_performance['client'] == 1]
# Divide datadisk by the count of rows with the same type and num_tenants
df = df_performance_first.copy()
# Create a mask for rows where type is not "container"
mask = df['type'] != 'container'

# Only apply the group count to the relevant rows
group_counts = df[mask].groupby(['type', 'num_tenants'])['datadisk'].transform('count')

# Initialize the column with NaN (or 0, if preferred)
df['datadisk_normalized'] = df['datadisk'] / 1024

# Apply the normalized value only where the mask is True
df.loc[mask, 'datadisk_normalized'] = df.loc[mask, 'datadisk'] / group_counts / 1024

plot_bars(df, y='datadisk_normalized', title='Database Size [GB]', estimator='sum', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance_first, y='datadisk', title='Database Size [MB]', estimator='sum', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance_first, y='time_ingest', title='Time for Ingestion [s]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance_first, y='time_check', title='Time for Vacuum [s]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

# Hardware Monitoring for Benchmarking Phase

In [ ]:
df_performance = collect.get_monitoring_all(type="stream")

df_performance_first = df_performance[df_performance['client']==1]
df_performance_second = df_performance[df_performance['client']==2]

df_performance.T

In [ ]:
plot_bars(df_performance, y='CPU Utilization Time [s]', title='CPU [CPUs]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance_first, y='CPU Utilization Time [s]', title='CPU 1st Run [CPUs]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance_second, y='CPU Utilization Time [s]', title='CPU 2nd Run [CPUs]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance, y='CPU Utilization', title='CPU Utilization [%]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

## Custom Aggregation

In [ ]:
metric = 'total_cpu_util'

df_performance_series = collect.get_monitoring_timeseries_all(metric)

df_agg = (
    df_performance_series.groupby(["client", "type", "num_tenants"])["value"]
      .max()
      .reset_index()
)
plot_bars(df_agg, y='value', title='Max CPU Utilization [%]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)
#df_agg

In [ ]:
plot_bars(df_performance, y='CPU Throttle', title='CPU Throttle [%]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

## Custom Aggregation and Scale

In [ ]:
metric = 'total_cpu_memory'

df_performance_series = collect.get_monitoring_timeseries_all(metric)

df_agg = (
    df_performance_series.groupby(["client", "type", "num_tenants"])["value"]
      .max()
      .reset_index()
)
df_agg['value'] = df_agg['value'] / 1024.
plot_bars(df_agg, y='value', title='Max Memory Usage [GiB]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

plot_bars(df_performance, y='Memory Usage [MiB]', title='Average Memory Usage [MiB]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
metric = 'total_cpu_memory_cached'

df_performance_series = collect.get_monitoring_timeseries_all(metric)

df_agg = (
    df_performance_series.groupby(["client", "type", "num_tenants"])["value"]
      .max()
      .reset_index()
)
df_agg['value'] = df_agg['value'] / 1024.
plot_bars(df_agg, y='value', title='Max Memory Usage Cached [GiB]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

plot_bars(df_performance, y='Memory Usage Cached [MiB]', title='Average Memory Usage Cached [MiB]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

# Hardware Monitoring for Loading Phase

In [ ]:
df_performance = collect.get_monitoring_all("loading")

df_performance_first = df_performance[df_performance['client']==1]
df_performance_second = df_performance[df_performance['client']==2]

df_performance_first

In [ ]:
plot_bars(df_performance, y='CPU Utilization Time [s]', title='CPU [CPUs]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance, y='Memory Usage [MiB]', title='Memory Usage [MiB]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(df_performance, y='Memory Usage Cached [MiB]', title='Memory Usage Cached [MiB]', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

# Efficiency

In [ ]:
client = 1

df_performance_monitoring = collect.get_monitoring_all(type="stream")
df_performance_monitoring = df_performance_monitoring[df_performance_monitoring['client'] == client]
df_performance = collect.get_performance_all()
df_performance = df_performance[df_performance['client'] == client]
merged_df = pd.merge(df_performance, df_performance_monitoring, on=['type', 'num_tenants', 'code', 'client'], how='inner')
#merged_df['I_Lat'] = 1./merged_df['E_Lat']
merged_df['E_Tpx'] = merged_df['Goodput (requests/second)'] / merged_df['CPU Utilization Time [s]'] * 600.
merged_df['E_Lat'] = 1./np.sqrt(merged_df['Latency Distribution.Average Latency (microseconds)']*merged_df['CPU Utilization Time [s]']/1E6)
merged_df['E_RAM'] = (merged_df['Goodput (requests/second)']) / merged_df['Memory Usage [MiB]']
merged_df

In [ ]:
plot_bars(merged_df, y='E_Lat', title='1st run - $E_{Lat}$', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
#plot_bars(merged_df, y='I_Lat', title='1st run - $I_{Lat}$', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(merged_df, y='E_Tpx', title='1st run - $E_{Tpx}$', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(merged_df, y='E_RAM', title='1st run - $E_{RAM}$', estimator='min', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
client = 2

df_performance_monitoring = collect.get_monitoring_all(type="stream")
df_performance_monitoring = df_performance_monitoring[df_performance_monitoring['client'] == client]
df_performance = collect.get_performance_all()
df_performance = df_performance[df_performance['client'] == client]
merged_df = pd.merge(df_performance, df_performance_monitoring, on=['type', 'num_tenants', 'code', 'client'], how='inner')
#merged_df['CPUs/Request'] = merged_df['CPU [CPUs]'] / merged_df['Goodput (requests/second)'] / 600.
merged_df['E_Tpx'] = merged_df['Goodput (requests/second)'] / merged_df['CPU Utilization Time [s]'] * 600.
merged_df['E_Lat'] = 1./np.sqrt(merged_df['Latency Distribution.Average Latency (microseconds)']*merged_df['CPU Utilization Time [s]']/1E6)
merged_df['E_RAM'] = (merged_df['Goodput (requests/second)']) / merged_df['Memory Usage [MiB]']

merged_df

In [ ]:
plot_bars(merged_df, y='E_Lat', title='2nd run - $E_{Lat}$', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(merged_df, y='E_Tpx', title='2nd run - $E_{Tpx}$', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
plot_bars(merged_df, y='E_RAM', title='2nd run - $E_{RAM}$', estimator='max', b_plot_save=b_plot_save, filename_prefix=filename_prefix)

In [ ]:
#zip_all_results()